In [ ]:
%pip install dotenv
%pip install datasets
%pip install scikit-learn
%pip install -r requirements.txt
%load_ext autoreload
%autoreload 2


In [ ]:
import sys

import torch

sys.path.append(".")


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CPU cores:", os.cpu_count())


In [ ]:
from huggingface_hub import login
import os
from dotenv import load_dotenv

load_dotenv()

# Use your token to log in
login(token=os.getenv("hf_token"))


# AllSides → MITweet

Does fine-tuning on AllSides news bias buy anything on MITweet tweets? MITweet's `I1`-`I12`
are `0=left / 1=center / 2=right`, exactly the AllSides `bias` labels, so an
AllSides-fine-tuned `bias_head` transfers with no weight surgery at all.

Three arms, all starting from the same AllSides fine-tune:

1. **AllSides only** — fine-tune on AllSides, then predict MITweet ideology with no MITweet
   training whatsoever.
2. **AllSides → MITweet ideology** — fine-tune on AllSides, then fine-tune again on MITweet.
3. **AllSides → MITweet relevance** — same, for the relevance task. The relevance head is new
   here; only the backbone carries the AllSides adaptation.

Compare against the MITweet-only numbers from `run-mitweet-experiments.ipynb`, which is arm 2
without its first stage.

In [ ]:
import glob
from pathlib import Path

from huggingface_hub import snapshot_download

from config import load_run_config
from finetuning import aggregate as agg
from finetuning.experiments import (
    run_experiment, run_mitweet_experiment, run_prediction_only,
    DatasetConfig, ExperimentConfig,
    ALLSIDES_BASE_MEDIA_SPLIT, ALLSIDES_BASE_RANDOM_SPLIT,
)
from finetuning.mitweet import MITweetConfig, variant_name
from finetuning.models import BERT, BART, ROBERTA, POLITICS, IDEOLOGY_CLASSIFIER


In [ ]:
ideology_dir = snapshot_download(repo_id=IDEOLOGY_CLASSIFIER)
ideology_pt = glob.glob(f"{ideology_dir}/*.pt")[0]
print(f"Ideology classifier .pt: {ideology_pt}")


In [ ]:
def find_tlp_checkpoints(config_glob="run_configs/tlp_*.yaml"):
    """Locate the checkpoint each tlp_* pretraining run left behind.
    """
    checkpoints = []
    for config_path in sorted(glob.glob(config_glob)):
        label = Path(config_path).stem
        output_dir = Path(load_run_config(config_path).output_dir)
        epochs = sorted(
            output_dir.glob("epoch-*.pt"),
            key=lambda p: int(p.stem.split("-")[1]),
        )
        if not epochs:
            print(f"  SKIP {label}: no epoch-*.pt under {output_dir}")
            continue
        print(f"  {label}: {epochs[-1]}")
        checkpoints.append((str(epochs[-1]), label))
    return checkpoints


print("tlp checkpoints:")
TLP_CHECKPOINTS = find_tlp_checkpoints()
print(f"\nfound {len(TLP_CHECKPOINTS)} of 4")


In [ ]:
BASELINES = [
    (BERT, "bert"),
    (BART, "bart"),
    (ROBERTA, "roberta"),
    (POLITICS, "politics"),
    (ideology_pt, "ideology"),
]
MODELS = BASELINES + TLP_CHECKPOINTS

seeds = [42, 1, 13, 1234, 6789]
# The media split keeps every outlet inside one of train/test; it is the harder
# comparison, and the one the AllSides numbers are reported on.
ALLSIDES_SPLITS = {
    "media_split": ALLSIDES_BASE_MEDIA_SPLIT,
    "random_split": ALLSIDES_BASE_RANDOM_SPLIT,
}
ALLSIDES_DATASET = ALLSIDES_SPLITS["media_split"]
ROOT = "results_allsides_to_mitweet"

# Which MITweet variant the transfer arms are scored on.
IDEOLOGY_VARIANT = MITweetConfig(task="ideology", prepend="indicators", split="random")
RELEVANCE_VARIANT = MITweetConfig(task="relevance", prepend="none", split="random")


## Stage 1: fine-tune on AllSides, and keep the checkpoint

`save_model=True` writes the best epoch somewhere `load_model` can read back. For a
`MultiTaskRoberta` that is a `.pt` written by `save_checkpoint` (which records the head
sizes); for an HF baseline it is a `pytorch_model.bin` directory. The path comes back in the
metrics as `saved_model_path`.

This is the expensive cell — every MITweet arm below reuses its checkpoints.

In [ ]:
def finetune_on_allsides(models, dataset, seed, root=ROOT):
    """Fine-tune each model on AllSides once, returning (checkpoint_path, name) pairs.
    """
    loc = f"{root}/allsides/seed_{seed}"
    os.makedirs(loc, exist_ok=True)
    checkpoints = []
    for model_ref, model_name in models:
        print(f"\n{'='*60}\nAllSides: {model_name}  |  Seed: {seed}\n{'='*60}")
        metrics = run_experiment(
            model=model_ref,
            loc=loc,
            dataset_config=DatasetConfig(custom_dataset=dataset),
            experiment_config=ExperimentConfig(
                patience=3, num_epochs=15, save_model=True, seed=seed
            ),
            model_name=model_name,
        )
        checkpoints.append((metrics["saved_model_path"], model_name))
    return checkpoints


ALLSIDES_CHECKPOINTS = {
    seed: finetune_on_allsides(MODELS, ALLSIDES_DATASET, seed) for seed in seeds
}


## Arm 1: zero-shot on MITweet

No MITweet training. The AllSides `bias_head` is asked the ideology question directly.

One caveat to read the prefixed variants with: an AllSides model has never seen a
`<prefix> </s></s> <tweet>` input, so `prepend="indicators"` is out of distribution for it in
a way it is not for a MITweet-trained model. `prepend="none"` is the clean zero-shot reading;
the prefixed arms say what the head does with an unfamiliar input shape.

In [ ]:
def predict_zero_shot(checkpoints, mitweet_config, seed, root=ROOT):
    """Score each AllSides checkpoint on one MITweet test split without training it.
    """
    loc = f"{root}/zeroshot_{variant_name(mitweet_config)}/seed_{seed}"
    os.makedirs(loc, exist_ok=True)
    results = {}
    for path, model_name in checkpoints:
        print(f"\n{'='*60}\nzero-shot: {model_name}  |  Seed: {seed}\n{'='*60}")
        results[model_name] = run_prediction_only(
            model=path,
            loc=loc,
            mitweet_config=mitweet_config,
            model_name=model_name,
            seed=seed,
        )
    return results


for seed in seeds:
    for prepend in ["none", "indicators", "facet_name"]:
        predict_zero_shot(
            ALLSIDES_CHECKPOINTS[seed],
            MITweetConfig(task="ideology", prepend=prepend, split="random"),
            seed,
        )


## Arms 2 and 3: AllSides, then MITweet

The same checkpoints, fine-tuned a second time. Arm 2 continues training the very
`bias_head` AllSides trained; arm 3 attaches a fresh 12-way relevance head to an
AllSides-adapted backbone.

In [ ]:
def finetune_from_allsides(checkpoints, mitweet_config, seed, root=ROOT):
    """Fine-tune each AllSides checkpoint on one MITweet variant.
    """
    loc = f"{root}/sequential_{variant_name(mitweet_config)}/seed_{seed}"
    os.makedirs(loc, exist_ok=True)
    results = {}
    for path, model_name in checkpoints:
        print(f"\n{'='*60}\nAllSides -> {variant_name(mitweet_config)}: {model_name}  |  Seed: {seed}\n{'='*60}")
        results[model_name] = run_mitweet_experiment(
            model=path,
            loc=loc,
            mitweet_config=mitweet_config,
            experiment_config=ExperimentConfig(
                patience=2, num_epochs=10, save_model=False, seed=seed
            ),
            model_name=model_name,
        )
    return results


for seed in seeds:
    finetune_from_allsides(ALLSIDES_CHECKPOINTS[seed], IDEOLOGY_VARIANT, seed)
    finetune_from_allsides(ALLSIDES_CHECKPOINTS[seed], RELEVANCE_VARIANT, seed)


## Cross-seed analysis

Reads the per-seed JSONs off disk. To answer "did AllSides help?", compare
`sequential_ideology_*` here against the same variant in `results_mitweet/` from the other
notebook — same models, same seeds, same test rows, one extra training stage.

In [ ]:
import pandas as pd

IDEOLOGY_METRICS = ["f1_macro", "accuracy", "facet_f1_macro", "facet_accuracy"]

rows = []
for variant in sorted(os.listdir(ROOT)):
    runs = agg.discover_results(f"{ROOT}/{variant}")
    if not runs:
        continue
    metrics = ["f1_macro", "f1_micro"] if "relevance" in variant else IDEOLOGY_METRICS
    for model, row in agg.summary_table(runs, metrics=metrics).iterrows():
        rows.append({"arm": variant, "model": model, **row.to_dict()})

pd.DataFrame(rows).set_index(["arm", "model"])


In [ ]:
# Paired per-seed delta: MITweet-only vs AllSides-then-MITweet, on the same variant.
# Unpaired seeds are dropped loudly -- averaging one in as a "delta" would be a wrong number.
MITWEET_ONLY = f"results_mitweet/{variant_name(IDEOLOGY_VARIANT)}"
SEQUENTIAL = f"{ROOT}/sequential_{variant_name(IDEOLOGY_VARIANT)}"

direct = agg.by_model(agg.discover_results(MITWEET_ONLY))
transfer = agg.by_model(agg.discover_results(SEQUENTIAL))

rows, unpaired = [], []
for model, runs in sorted(transfer.items()):
    baseline_by_seed = {r.seed: r.metrics for r in direct.get(model, [])}
    for run in runs:
        if run.seed not in baseline_by_seed:
            unpaired.append(f"{model} seed {run.seed}")
            continue
        for metric in IDEOLOGY_METRICS:
            rows.append({
                "model": model,
                "metric": metric,
                "seed": run.seed,
                "delta": run.metrics[metric] - baseline_by_seed[run.seed][metric],
            })

if unpaired:
    print("dropped, no MITweet-only run at the same seed:", ", ".join(unpaired))

frame = pd.DataFrame(rows)
frame.groupby(["model", "metric"])["delta"].agg(
    mean_delta="mean", std=lambda s: s.std(ddof=1), n_better=lambda s: int((s > 0).sum()), n="count"
).round(2)
